# 05 · Alineación (versión rápida)

Usa el **registro de predicciones** (`data/predictions_log/`, notebook 04) para:
1. mostrar todo mi roster con la predicción del modelo, la proyección de ESPN y el estado de lesión;
2. armar la alineación óptima con los slots de la liga y compararla con mi alineación actual;
3. sugerir los 5 mejores agentes libres por posición.

**De dónde sale cada dato:**
- **Predicciones y proyecciones de ESPN:** del registro. Es la última predicción hecha antes del partido, así que son exactamente los números que quedaron guardados y fechados en git.
- **Roster, slots actuales, estado de lesión y agentes libres:** de ESPN **en vivo**, para decidir con la información más reciente.

**Umbral de 3 puntos:** el error típico del modelo es de unos 5–6 puntos por jugador y partido (notebook 03). Un cambio que gana menos de 3 puntos no es concluyente: los dos jugadores rinden prácticamente igual y la decisión puede depender de otros factores (clima, noticias de última hora, preferencia personal).

_Pendiente para la versión completa: rangos de predicción y análisis de varias semanas._

## 1. Datos

In [ ]:
import polars as pl

from fantasy_ml import espn, lineup as L, predictions_log as plog

SEASON = 2026
THRESHOLD = 3.0  # puntos: por debajo, el cambio no es concluyente

league = espn.connect(SEASON)
WEEK = league.current_week
SLOTS = {k: v for k, v in league.settings.position_slot_counts.items() if v}

log_all = plog.read(SEASON)
log = plog.latest_pregame(log_all).filter(pl.col("week") == WEEK)
assert log.height, f"No hay predicciones registradas para la semana {WEEK}: ejecuta el notebook 04"
print(f"Semana {WEEK} · slots de la liga: {SLOTS}")
print(f"Predicciones del registro: {log.height} · generadas {log['generated_at_utc'].max():%Y-%m-%d %H:%M} UTC "
      f"· versión del código {log['code_version'].unique().to_list()}")

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(12)
pl.Config.set_fmt_str_lengths(30)

## 2. Mi roster

Un jugador **no está disponible** si ESPN lo marca OUT, IR o suspendido, o si no tiene predicción (semana libre o inactivo en nflverse). *Questionable* y *doubtful* se muestran con ⚠, pero siguen elegibles.

In [ ]:
live = espn.rostered_projections(league, WEEK).filter("on_my_roster")
preds = log.select("espn_id", "pred_model", "espn_projection", "opponent", "kickoff_utc")

roster = (live.select("espn_id", name="espn_name", position="espn_position", my_slot="my_slot",
                      injury="espn_injury_status")
          .join(preds, on="espn_id", how="left")
          .with_columns(
              available=pl.col("pred_model").is_not_null() & ~pl.col("injury").fill_null("").is_in(list(L.UNAVAILABLE)),
              reason=pl.when(pl.col("pred_model").is_null()).then(pl.lit("sin predicción (bye/inactivo)"))
                       .when(pl.col("injury").is_in(list(L.UNAVAILABLE))).then(pl.col("injury"))
                       .otherwise(pl.lit("rendimiento")),
              aviso=pl.when(pl.col("injury").is_in(list(L.DOUBTFUL))).then(pl.lit("⚠ ") + pl.col("injury"))
                      .when(pl.col("injury").is_in(list(L.UNAVAILABLE))).then(pl.lit("✗ ") + pl.col("injury"))
                      .otherwise(pl.lit(""))))

SLOT_ORDER = {s: i for i, s in enumerate(["QB", "RB", "WR", "TE", "RB/WR/TE", "OP", "K", "D/ST", "BE", "IR"])}
(roster.with_columns(_o=pl.col("my_slot").replace_strict(SLOT_ORDER, default=99))
       .sort("_o", pl.col("pred_model"), descending=[False, True], nulls_last=True)
       .select(slot_actual="my_slot", jugador="name", pos="position", rival="opponent", lesion="injury", aviso="aviso",
               modelo=pl.col("pred_model").round(1), espn="espn_projection",
               modelo_menos_espn=(pl.col("pred_model") - pl.col("espn_projection")).round(1)))

## 3. Alineación óptima

Se asignan los mejores jugadores disponibles a cada slot: primero los fijos (QB, RB, WR, TE, K y D/ST) y después los FLEX (RB/WR/TE). Hago lo mismo con las proyecciones de ESPN para ver si ESPN recomendaría los mismos cambios.

In [ ]:
opt_model = L.optimal_lineup(roster, SLOTS, "pred_model")
opt_espn = L.optimal_lineup(roster.with_columns(available=pl.col("available") & pl.col("espn_projection").is_not_null()),
                            SLOTS, "espn_projection")

show = roster.select("espn_id", "name", "position", "my_slot", "pred_model", "espn_projection", "aviso")
lineup_table = (opt_model.with_row_index("_i")
    .join(show, on="espn_id", how="left")
    .join(opt_espn.select("espn_id", espn_tambien=pl.lit("✓")), on="espn_id", how="left")
    .sort("_i")
    .select(slot="slot", jugador="name", pos="position", slot_actual="my_slot", aviso="aviso",
            modelo=pl.col("pred_model").round(1), espn="espn_projection", espn_lo_alinea=pl.col("espn_tambien").fill_null("✗")))

cur = roster.filter(~pl.col("my_slot").is_in(list(L.BENCH_SLOTS)))
print(f"Puntos proyectados por el modelo · alineación actual: {cur['pred_model'].fill_null(0).sum():.1f} "
      f"· óptima: {lineup_table['modelo'].fill_null(0).sum():.1f}")
lineup_table

## 4. Cambios recomendados

Solo cuenta **quién es titular**, no en qué slot: mover a un RB del slot RB al FLEX no es un cambio. Cada jugador que entra se empareja con uno que sale de su misma posición, o con el peor titular si el cambio pasa por el FLEX.

- **Concluyente:** el modelo gana 3 o más puntos, o el que sale no puede jugar (OUT/IR/bye).
- **No concluyente:** gana menos de 3 puntos. Los dos jugadores rinden prácticamente igual.

In [ ]:
changes = L.lineup_changes(roster, opt_model, "pred_model", THRESHOLD)
espn_starters = set(opt_espn["espn_id"].drop_nulls().to_list())
name_to_id = dict(zip(roster["name"], roster["espn_id"]))

if changes.is_empty():
    print("✓ Tu alineación actual ya es la óptima según el modelo.")
else:
    changes = changes.with_columns(
        veredicto=pl.when("concluyente").then(pl.lit("✓ hacer el cambio"))
                    .otherwise(pl.lit(f"≈ no concluyente (< {THRESHOLD:g} pts)")),
        espn_de_acuerdo=pl.col("entra").map_elements(lambda n: "sí" if name_to_id[n] in espn_starters else "no",
                                                      return_dtype=pl.Utf8))
    display(changes.select("veredicto", "entra", "pos_entra", "sale", "pos_sale", "motivo_salida",
                           "pred_entra", "pred_sale", "diferencia", "espn_de_acuerdo"))

## 5. Mejores agentes libres por posición

Top 5 por predicción del modelo entre los agentes libres de ESPN **en este momento** que tienen predicción en el registro. Los que no la tienen no aparecen en los rosters activos de nflverse (lesionados, practice squad, etc.).

**`mejora_alineacion`:** cuántos puntos sube mi alineación óptima si agrego a ese jugador. Es 0 si no entraría de titular. Es mejor que compararlo con el peor jugador de su posición, porque tiene en cuenta los FLEX: un WR nuevo puede desplazar al RB más débil del FLEX, no al WR de la banca. Se aplica el mismo umbral de 3 puntos.

In [ ]:
fa = (espn.free_agent_projections(league, WEEK)
      .select("espn_id", name="espn_name", position="espn_position", injury="espn_injury_status")
      .join(preds, on="espn_id", how="inner")
      .filter(~pl.col("injury").fill_null("").is_in(list(L.UNAVAILABLE))))

top_fa = fa.with_columns(available=pl.lit(True)).sort("pred_model", descending=True).group_by("position", maintain_order=True).head(5)
top_fa = (top_fa.with_columns(L.pickup_gain(roster, top_fa, SLOTS, "pred_model").round(1))
            .with_columns(_o=pl.col("position").replace_strict(SLOT_ORDER, default=99),
                          veredicto=pl.when(pl.col("mejora_alineacion") >= THRESHOLD).then(pl.lit("✓ vale la pena"))
                                      .when(pl.col("mejora_alineacion") > 0).then(pl.lit(f"≈ no concluyente (< {THRESHOLD:g} pts)"))
                                      .otherwise(pl.lit("no entraría de titular")))
            .sort("_o", pl.col("pred_model"), descending=[False, True])
            .select(pos="position", jugador="name", rival="opponent", lesion="injury",
                    modelo=pl.col("pred_model").round(1), espn="espn_projection", mejora_alineacion="mejora_alineacion",
                    veredicto="veredicto"))
print(f"Agentes libres con predicción: {fa.height}")
top_fa